# 02b — Build Pinecone Vector Store

Encode all chunks with the best embedding model and upsert to Pinecone.

In [ ]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.embeddings.vector_store import VectorStore

## 1. Load chunks

In [ ]:
chunks_df = pd.read_parquet(PROJECT_ROOT / "data/processed/medical_chunks.parquet")
chunks = chunks_df.to_dict("records")
print(f"Loaded {len(chunks)} chunks")

## 2. Build index

Using the winning model from the embedding comparison (default: PubMedBert).

In [ ]:
store = VectorStore(model_name="pritamdeka/S-PubMedBert-MS-MARCO")
total = store.build_index(chunks)
print(f"Upserted {total} vectors")

## 3. Verify with a test search

In [ ]:
results = store.search("What are the symptoms of heart failure?", top_k=5)
for r in results:
    print(f"[{r['score']:.3f}] {r.get('text', '')[:120]}...")

## 4. Test content recommender

In [ ]:
from src.embeddings.recommender import ContentRecommender

recommender = ContentRecommender(store)
recs = recommender.recommend_study_path(["heart failure symptoms", "diabetes treatment"])
for topic, items in recs.items():
    print(f"\n{topic}:")
    for item in items:
        print(f"  [{item['score']:.3f}] {item.get('text', '')[:100]}...")